# Synthetic Tests of Independence of Irrelevant Alternatives

## Objective

Study how often two likelihood-ratio tests reject the MNL model
when observations are generated by:

1. An MNL model, where IIA holds.
2. A full CDM, which can represent context-dependent choices.
3. A general choice system with separately specified probabilities
   for each choice set.

We compare MNL against two alternatives: full CDM and the
universal choice model.

The experiment reports rejection proportions across repeated
datasets as the number of observations increases.

In [1]:
# Import dependencies and locate the project and reusable model modules.

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from IPython.display import display

import cdm_paper_reproduction.estimation as paper_estimation
import cdm_paper_reproduction.synthetic as paper_synthetic
import context_dependent_model as cdm

current_directory = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        directory
        for directory in (
            current_directory,
            *current_directory.parents,
        )
        if (directory / "pyproject.toml").is_file()
        and (directory / "src").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root containing pyproject.toml and src."
    )

print("Project root:", PROJECT_ROOT)
print("Synthetic module:", paper_synthetic.__file__)
print("Estimation module:", paper_estimation.__file__)

Project root: /Users/zsres/Documents/M.Tech/Nitesh-Kumar-Shah-MTP-
Synthetic module: /Users/zsres/Documents/M.Tech/Nitesh-Kumar-Shah-MTP-/src/cdm_paper_reproduction/synthetic.py
Estimation module: /Users/zsres/Documents/M.Tech/Nitesh-Kumar-Shah-MTP-/src/cdm_paper_reproduction/estimation.py


In [2]:
# Configure the synthetic IIA pilot and create its output directories.

SEED = 20260831
NUMBER_OF_ITEMS = 6
SIGNIFICANCE_LEVEL = 0.05
PILOT_NUMBER_OF_RUNS = 10

OBSERVATION_COUNTS = (
    100,
    200,
    400,
    600,
    800,
    1_000,
    1_500,
    2_000,
)

DEVICE = torch.device("cpu")
DTYPE = torch.float64

GENERATING_MODELS = (
    "MNL",
    "CDM",
    "General choice",
)

EXPERIMENT_NAME = "02_synthetic_iia_tests"
experiment_path = (
    Path("cdm_paper_reproduction") / EXPERIMENT_NAME
)

FIGURE_DIRECTORY = (
    PROJECT_ROOT
    / "docs"
    / "thesis"
    / "images"
    / "experiments"
    / experiment_path
)

TABLE_DIRECTORY = (
    PROJECT_ROOT
    / "docs"
    / "thesis"
    / "tables"
    / "experiments"
    / experiment_path
)

RESULT_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "experiments"
    / experiment_path
)

for directory in (
    FIGURE_DIRECTORY,
    TABLE_DIRECTORY,
    RESULT_DIRECTORY,
):
    directory.mkdir(parents=True, exist_ok=True)

iia_settings = pd.DataFrame({
    "Setting": [
        "Random seed",
        "Number of items",
        "Significance level",
        "Pilot repetitions",
        "Observation counts",
        "Generating models",
        "Device",
        "Dtype",
    ],
    "Value": [
        SEED,
        NUMBER_OF_ITEMS,
        SIGNIFICANCE_LEVEL,
        PILOT_NUMBER_OF_RUNS,
        str(OBSERVATION_COUNTS),
        ", ".join(GENERATING_MODELS),
        str(DEVICE),
        str(DTYPE),
    ],
})

display(iia_settings)

iia_settings.to_csv(
    RESULT_DIRECTORY / "pilot_settings.csv",
    index=False,
)

,Setting,Value
0,Random seed,20260831
1,Number of items,6
2,Significance level,0.05
3,Pilot repetitions,10
4,Observation counts,"(100, 200, 400, 600, 800, 1000, 1500, 2000)"
5,Generating models,"MNL, CDM, General choice"
6,Device,cpu
7,Dtype,torch.float64


In [3]:
# Enumerate all valid choice sets for the six-item experiment.

from cdm_paper_reproduction.iia_simulation import (
    enumerate_choice_sets,
    generate_general_choice_probabilities,
    sample_choices_from_probability_table,
)

catalogue_choice_sets, catalogue_choice_mask = enumerate_choice_sets(
    number_of_items=NUMBER_OF_ITEMS,
    device=DEVICE,
)

catalogue_sizes = catalogue_choice_mask.sum(dim=1)

catalogue_summary = (
    pd.Series(
        catalogue_sizes.cpu().numpy(),
        name="Choice-set size",
    )
    .value_counts()
    .sort_index()
    .rename("Number of possible menus")
    .reset_index()
)

assert len(catalogue_choice_sets) == (
    2**NUMBER_OF_ITEMS - NUMBER_OF_ITEMS - 1
)

display(catalogue_summary)

print("Catalogue shape:", tuple(catalogue_choice_sets.shape))
print("Total possible menus:", len(catalogue_choice_sets))
print("\nFirst five padded menus:")
print(catalogue_choice_sets[:5])

,Choice-set size,Number of possible menus
0,2,15
1,3,20
2,4,15
3,5,6
4,6,1


Catalogue shape: (57, 6)
Total possible menus: 57

First five padded menus:
tensor([[ 0,  1, -1, -1, -1, -1],
        [ 0,  2, -1, -1, -1, -1],
        [ 0,  3, -1, -1, -1, -1],
        [ 0,  4, -1, -1, -1, -1],
        [ 0,  5, -1, -1, -1, -1]])


In [4]:
# Construct fixed probability tables for the three generating models.

MNL_PARAMETER_CONVENTION = "positive_luce_weights"
CDM_PARAMETER_SEED = SEED + 100
GENERAL_PARAMETER_SEED = SEED + 101

true_mnl_weights = torch.arange(
    1,
    NUMBER_OF_ITEMS + 1,
    device=DEVICE,
    dtype=DTYPE,
)
true_mnl_weights = true_mnl_weights / true_mnl_weights.sum()

true_mnl_log_utilities = true_mnl_weights.log()
true_mnl_log_utilities -= true_mnl_log_utilities.mean()

safe_catalogue_items = catalogue_choice_sets.clamp_min(0)

mnl_menu_weights = true_mnl_weights[safe_catalogue_items]
mnl_menu_weights = mnl_menu_weights.masked_fill(
    ~catalogue_choice_mask,
    0.0,
)

mnl_probability_table = (
    mnl_menu_weights
    / mnl_menu_weights.sum(dim=1, keepdim=True)
)

(
    iia_true_target_vectors,
    iia_true_context_vectors,
    iia_true_interactions,
) = paper_synthetic.generate_paper_cdm_parameters(
    number_of_items=NUMBER_OF_ITEMS,
    random_seed=CDM_PARAMETER_SEED,
    device=DEVICE,
    dtype=DTYPE,
)

with torch.no_grad():
    cdm_probability_table = cdm.cdm_choice_probabilities(
        interaction_matrix=iia_true_interactions,
        choice_sets=catalogue_choice_sets,
        choice_mask=catalogue_choice_mask,
    )

general_probability_table = generate_general_choice_probabilities(
    choice_mask=catalogue_choice_mask,
    random_seed=GENERAL_PARAMETER_SEED,
    dtype=DTYPE,
)

true_probability_tables = {
    "MNL": mnl_probability_table,
    "CDM": cdm_probability_table,
    "General choice": general_probability_table,
}

display(
    pd.DataFrame({
        "Item": range(NUMBER_OF_ITEMS),
        "True Luce weight": true_mnl_weights.cpu().numpy(),
        "Centred log-utility": (
            true_mnl_log_utilities.cpu().numpy()
        ),
    }).style.format({
        "True Luce weight": "{:.4f}",
        "Centred log-utility": "{:.4f}",
    })
)

,Item,True Luce weight,Centred log-utility
0,0,0.0476,-1.0965
1,1,0.0952,-0.4034
2,2,0.1429,0.0021
3,3,0.1905,0.2898
4,4,0.2381,0.5129
5,5,0.2857,0.6952


In [5]:
# Validate the probability tables and compare the same three-item menu.

probability_check_rows = []

for model_name, probabilities in true_probability_tables.items():
    assert torch.isfinite(probabilities).all().item()
    assert (probabilities >= 0).all().item()
    assert (probabilities[~catalogue_choice_mask] == 0).all().item()

    row_sum_error = (
        probabilities.sum(dim=1) - 1.0
    ).abs().max().item()

    assert row_sum_error < 1e-10

    probability_check_rows.append({
        "Generating model": model_name,
        "Menus": len(probabilities),
        "Maximum probability-sum error": row_sum_error,
        "Minimum valid probability": (
            probabilities[catalogue_choice_mask].min().item()
        ),
    })

display(pd.DataFrame(probability_check_rows))

example_menu = (0, 1, 2)

example_menu_index = next(
    row
    for row in range(len(catalogue_choice_sets))
    if tuple(
        catalogue_choice_sets[row][
            catalogue_choice_mask[row]
        ].cpu().tolist()
    ) == example_menu
)

example_probability_table = pd.DataFrame({
    "Item": example_menu,
    **{
        model_name: probabilities[
            example_menu_index,
            catalogue_choice_mask[example_menu_index],
        ].cpu().numpy()
        for model_name, probabilities
        in true_probability_tables.items()
    },
})

display(
    example_probability_table.style.format({
        model_name: "{:.4f}"
        for model_name in GENERATING_MODELS
    })
)

,Generating model,Menus,Maximum probability-sum error,Minimum valid probability
0,MNL,57,2.220446e-16,0.047619
1,CDM,57,2.220446e-16,0.000017
2,General choice,57,2.220446e-16,0.001654


,Item,MNL,CDM,General choice
0,0,0.1667,0.0024,0.5976
1,1,0.3333,0.9974,0.2397
2,2,0.5000,0.0002,0.1627


In [6]:
# Generate and save three demonstration datasets on the same sampled menus.

import json

DEMO_NUMBER_OF_OBSERVATIONS = max(OBSERVATION_COUNTS)
DEMO_MENU_SEED = SEED + 200

demo_outcome_seeds = {
    "MNL": SEED + 301,
    "CDM": SEED + 302,
    "General choice": SEED + 303,
}

menu_generator = torch.Generator(device="cpu")
menu_generator.manual_seed(DEMO_MENU_SEED)

demo_menu_indices = torch.randint(
    low=0,
    high=len(catalogue_choice_sets),
    size=(DEMO_NUMBER_OF_OBSERVATIONS,),
    generator=menu_generator,
).to(DEVICE)

demo_datasets = {}

for model_name, probabilities in true_probability_tables.items():
    demo_datasets[model_name] = (
        sample_choices_from_probability_table(
            catalogue_choice_sets=catalogue_choice_sets,
            catalogue_choice_mask=catalogue_choice_mask,
            probability_table=probabilities,
            menu_indices=demo_menu_indices,
            random_seed=demo_outcome_seeds[model_name],
        )
    )

demo_preview = pd.DataFrame({
    "Choice set": [
        tuple(
            catalogue_choice_sets[index][
                catalogue_choice_mask[index]
            ].cpu().tolist()
        )
        for index in demo_menu_indices[:10].cpu().tolist()
    ],
    **{
        f"{model_name} choice": data.chosen_items[:10].cpu().numpy()
        for model_name, data in demo_datasets.items()
    },
})

display(demo_preview)

for model_name, data in demo_datasets.items():
    chosen_is_available = (
        data.choice_sets.eq(data.chosen_items.unsqueeze(1))
        & data.choice_mask
    ).any(dim=1)

    assert chosen_is_available.all().item()

    model_filename = model_name.lower().replace(" ", "_")

    np.savez_compressed(
        RESULT_DIRECTORY / f"demo_{model_filename}.npz",
        choice_sets=data.choice_sets.cpu().numpy(),
        choice_mask=data.choice_mask.cpu().numpy(),
        chosen_positions=data.chosen_positions.cpu().numpy(),
        probability_table=(
            true_probability_tables[model_name].cpu().numpy()
        ),
    )

np.savez_compressed(
    RESULT_DIRECTORY / "demo_generating_parameters.npz",
    catalogue_choice_sets=catalogue_choice_sets.cpu().numpy(),
    catalogue_choice_mask=catalogue_choice_mask.cpu().numpy(),
    menu_indices=demo_menu_indices.cpu().numpy(),
    mnl_weights=true_mnl_weights.cpu().numpy(),
    mnl_log_utilities=true_mnl_log_utilities.cpu().numpy(),
    cdm_interactions=iia_true_interactions.cpu().numpy(),
)

demo_settings = {
    "number_of_items": NUMBER_OF_ITEMS,
    "number_of_observations": DEMO_NUMBER_OF_OBSERVATIONS,
    "mnl_parameter_convention": MNL_PARAMETER_CONVENTION,
    "cdm_parameter_seed": CDM_PARAMETER_SEED,
    "general_parameter_seed": GENERAL_PARAMETER_SEED,
    "menu_seed": DEMO_MENU_SEED,
    "outcome_seeds": demo_outcome_seeds,
    "menu_sampling": "uniform over all subsets of size at least two",
}

(RESULT_DIRECTORY / "demo_settings.json").write_text(
    json.dumps(demo_settings, indent=2),
    encoding="utf-8",
)

print(
    f"Generated {DEMO_NUMBER_OF_OBSERVATIONS:,} observations "
    "for each of the three models."
)
print("Every chosen item belongs to its observed menu.")
print("Saved datasets and generating settings:", RESULT_DIRECTORY)

,Choice set,MNL choice,CDM choice,General choice choice
0,"(0, 2, 3)",2,0,2
1,"(0, 2, 3, 4)",4,0,2
2,"(1, 3, 4, 5)",3,4,5
3,"(0, 4, 5)",5,5,5
4,"(2, 3, 5)",5,5,3
5,"(2, 4, 5)",5,2,4
6,"(1, 4, 5)",4,4,5
7,"(2, 3, 4, 5)",4,4,4
8,"(1, 4, 5)",4,4,4
9,"(0, 2, 4, 5)",5,5,4


Generated 2,000 observations for each of the three models.
Every chosen item belongs to its observed menu.
Saved datasets and generating settings: /Users/zsres/Documents/M.Tech/Nitesh-Kumar-Shah-MTP-/results/experiments/cdm_paper_reproduction/02_synthetic_iia_tests


## Fit MNL and Full CDM to Each Generated Dataset

For each generating model, fit MNL and full CDM to the same observed
choices. Initialize the full CDM using the fitted MNL probabilities.

Compare maximized training likelihoods and inspect optimization diagnostics.
These fits will subsequently be used for likelihood-ratio tests of IIA.

All observations are used for fitting here because the likelihood-ratio
comparison evaluates both models on the same sample.

In [7]:
# Fit an unregularized MNL model to each generated dataset.

from bradley_terry.choix_estimation import (
    estimate_top1_parameters_with_choix,
)

mnl_fits = {}

for generating_model, data in demo_datasets.items():
    observations = [
        (
            int(chosen_item),
            tuple(int(item) for item in choice_set[mask].tolist()),
        )
        for chosen_item, choice_set, mask in zip(
            data.chosen_items,
            data.choice_sets,
            data.choice_mask,
        )
    ]

    estimated_parameters = estimate_top1_parameters_with_choix(
        number_of_items=NUMBER_OF_ITEMS,
        observations=observations,
        alpha=0.0,
        maximum_iterations=10_000,
        tolerance=1e-10,
    )

    parameters = torch.as_tensor(
        estimated_parameters,
        dtype=DTYPE,
        device=data.choice_sets.device,
    )

    with torch.no_grad():
        utilities = parameters[
            data.choice_sets.clamp_min(0)
        ].masked_fill(~data.choice_mask, -torch.inf)

        chosen_utilities = utilities.gather(
            dim=1,
            index=data.chosen_positions.unsqueeze(1),
        ).squeeze(1)

        mean_nll = (
            torch.logsumexp(utilities, dim=1)
            - chosen_utilities
        ).mean().item()

    mnl_fits[generating_model] = {
        "parameters": parameters,
        "mean_nll": mean_nll,
        "number_of_observations": len(observations),
    }

    print(
        f"{generating_model} data | "
        f"MNL mean NLL = {mean_nll:.6f}"
    )

MNL data | MNL mean NLL = 1.035241
CDM data | MNL mean NLL = 0.861369
General choice data | MNL mean NLL = 1.116058


In [8]:
# Construct CDM initializations that reproduce the fitted MNL probabilities.

initial_cdm_interactions = {}
initialization_check_rows = []

for generating_model, data in demo_datasets.items():
    parameters = mnl_fits[generating_model]["parameters"]

    initial_interactions = (
        -parameters.unsqueeze(0)
        .expand(NUMBER_OF_ITEMS, -1)
        .clone()
    )

    initial_interactions = cdm.center_cdm_parameters(
        initial_interactions
    )

    with torch.no_grad():
        mnl_utilities = parameters[
            data.choice_sets.clamp_min(0)
        ].masked_fill(~data.choice_mask, -torch.inf)

        mnl_probabilities = torch.softmax(
            mnl_utilities,
            dim=1,
        )

        initial_cdm_probabilities = cdm.cdm_choice_probabilities(
            interaction_matrix=initial_interactions,
            choice_sets=data.choice_sets,
            choice_mask=data.choice_mask,
        )

        maximum_probability_difference = (
            mnl_probabilities - initial_cdm_probabilities
        ).abs().max().item()

    assert maximum_probability_difference < 1e-10, (
        f"MNL initialization mismatch for {generating_model}."
    )

    initial_cdm_interactions[generating_model] = (
        initial_interactions
    )

    initialization_check_rows.append({
        "Generating model": generating_model,
        "Maximum probability difference": (
            maximum_probability_difference
        ),
    })

initialization_check = pd.DataFrame(
    initialization_check_rows
)

display(
    initialization_check.style.format({
        "Maximum probability difference": "{:.2e}",
    })
)

,Generating model,Maximum probability difference
0,MNL,5.55e-17
1,CDM,1.11e-16
2,General choice,5.55e-17


In [9]:
# Fit the full CDM from the equivalent MNL initialization.

cdm_fits = {}
cdm_fit_rows = []

for generating_model, data in demo_datasets.items():
    print(f"Fitting full CDM to {generating_model} data...")

    result = paper_estimation.optimize_full_cdm_with_adam(
        number_of_items=NUMBER_OF_ITEMS,
        training_data=data,
        initial_interactions=(
            initial_cdm_interactions[generating_model]
        ),
        learning_rate=0.05,
        gradient_tolerance=1e-5,
        nll_tolerance=1e-9,
        patience=50,
        maximum_epochs=5_000,
        dtype=DTYPE,
    )

    with torch.no_grad():
        fitted_nll = cdm.average_cdm_negative_log_likelihood(
            interaction_matrix=result.interaction_matrix,
            choice_sets=data.choice_sets,
            chosen_positions=data.chosen_positions,
            choice_mask=data.choice_mask,
        ).item()

    cdm_fits[generating_model] = result

    cdm_fit_rows.append({
        "Generating model": generating_model,
        "CDM mean NLL": fitted_nll,
        "CDM epochs": result.epochs_completed,
        "CDM runtime (seconds)": result.runtime_seconds,
        "CDM converged": result.converged,
        "CDM stopping reason": result.stopping_reason,
    })

    print(
        f"  NLL = {fitted_nll:.6f} | "
        f"epochs = {result.epochs_completed} | "
        f"converged = {result.converged}"
    )

Fitting full CDM to MNL data...
  NLL = 1.028059 | epochs = 143 | converged = True
Fitting full CDM to CDM data...
  NLL = 0.364798 | epochs = 1254 | converged = True
Fitting full CDM to General choice data...
  NLL = 1.083702 | epochs = 153 | converged = True


In [10]:
# Compare fitted likelihoods and save the demo fitting results.

demo_fit_summary = pd.DataFrame(cdm_fit_rows)

demo_fit_summary["Observations"] = (
    demo_fit_summary["Generating model"].map({
        name: fit["number_of_observations"]
        for name, fit in mnl_fits.items()
    })
)

demo_fit_summary["MNL mean NLL"] = (
    demo_fit_summary["Generating model"].map({
        name: fit["mean_nll"]
        for name, fit in mnl_fits.items()
    })
)

demo_fit_summary["NLL improvement over MNL"] = (
    demo_fit_summary["MNL mean NLL"]
    - demo_fit_summary["CDM mean NLL"]
)

demo_fit_summary["CDM no worse than MNL"] = (
    demo_fit_summary["NLL improvement over MNL"] >= -1e-8
)

demo_fit_summary = demo_fit_summary[
    [
        "Generating model",
        "Observations",
        "MNL mean NLL",
        "CDM mean NLL",
        "NLL improvement over MNL",
        "CDM no worse than MNL",
        "CDM converged",
        "CDM epochs",
        "CDM runtime (seconds)",
        "CDM stopping reason",
    ]
]

RESULT_DIRECTORY.mkdir(parents=True, exist_ok=True)

demo_fit_summary.to_csv(
    RESULT_DIRECTORY / "demo_fit_summary.csv",
    index=False,
)

for generating_model, result in cdm_fits.items():
    model_slug = generating_model.lower().replace(" ", "_")

    np.savez_compressed(
        RESULT_DIRECTORY / f"demo_{model_slug}_fitted_parameters.npz",
        mnl_parameters=(
            mnl_fits[generating_model]["parameters"]
            .detach().cpu().numpy()
        ),
        initial_cdm_interactions=(
            initial_cdm_interactions[generating_model]
            .detach().cpu().numpy()
        ),
        fitted_cdm_interactions=(
            result.interaction_matrix.detach().cpu().numpy()
        ),
    )

display(
    demo_fit_summary.style.format({
        "MNL mean NLL": "{:.6f}",
        "CDM mean NLL": "{:.6f}",
        "NLL improvement over MNL": "{:.6f}",
        "CDM runtime (seconds)": "{:.3f}",
    })
)

if not demo_fit_summary["CDM no worse than MNL"].all():
    print("Inspect the affected CDM fits before calculating test statistics.")

if not demo_fit_summary["CDM converged"].all():
    print("Some CDM fits reached a stopping condition without convergence.")

,Generating model,Observations,MNL mean NLL,CDM mean NLL,NLL improvement over MNL,CDM no worse than MNL,CDM converged,CDM epochs,CDM runtime (seconds),CDM stopping reason
0,MNL,2000,1.035241,1.028059,0.007182,True,True,143,0.288,gradient tolerance reached
1,CDM,2000,0.861369,0.364798,0.496571,True,True,1254,1.806,gradient tolerance reached
2,General choice,2000,1.116058,1.083702,0.032356,True,True,153,0.217,gradient tolerance reached
